In [1]:
!pip freeze | grep scikit-learn

scikit-learn==1.6.1


In [1]:
!python -V

Python 3.12.1


In [2]:
import pickle
import pandas as pd

In [3]:
output_file = 'data/output_file.parquet'

In [4]:
with open('model.bin', 'rb') as f_in:
    dv, model = pickle.load(f_in)

/home/codespace/.local/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator DictVectorizer from version 1.5.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/codespace/.local/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LinearRegression from version 1.5.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [6]:
categorical = ['PULocationID', 'DOLocationID']

def read_data(filename):
    
    df = pd.read_parquet(filename)
    
    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df['duration'] = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)].copy()

    df[categorical] = df[categorical].fillna(-1).astype('int').astype('str')
    return df

In [7]:
year = 2023
month = 3
filename = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year:04d}-{month:02d}.parquet"
df = read_data(filename)

In [8]:
dicts = df[categorical].to_dict(orient='records')
X_val = dv.transform(dicts)
y_pred = model.predict(X_val)

In [18]:
# Q1. Notebook
y_pred_series = pd.Series(y_pred)
print("y prediction Standard Deviation : ", float(y_pred_series.describe().loc['std'].round(3)))

y prediction Standard Deviation :  6.247


In [22]:
y_pred[:5]

array([16.24590642, 26.1347962 , 11.88426424, 11.99771983, 10.23448579])

In [ ]:
#Q2. Preparing the output
def save_results(df, y_pred, output_file, year, month):

    df['ride_id'] = f'{year:04d}/{month:02d}_' + df.index.astype('str')

    df_result = pd.DataFrame()
    df_result['ride_id'] = df['ride_id']
    df_result['predicted_duration'] = y_pred

    df_result.to_parquet(
        output_file,
        engine='pyarrow',
        compression=None,
        index=False
    )
    print("New parquet file created: ", output_file)

In [30]:
save_results(df, y_pred, output_file, year, month)
!ls -lh $output_file

New parquet file created:  data/output_file.parquet
-rw-rw-rw- 1 codespace codespace 66M Jun 22 12:10 data/output_file.parquet


In [ ]:
#Q3. Creating the scoring script 